In [1]:
!pip install -q transformers datasets accelerate peft bitsandbytes wandb TRL

In [1]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: oktmadmom (oktmadmom-innovasoft) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [13]:
import os

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset
from transformers import AutoTokenizer

import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

In [12]:

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
TRAIN_PATH = "/home/Train_Data/train.jsonl"
TEST_PATH = "/home/Train_Data/test.jsonl"
OUTPUT_DIR = "./coffee_qwen_adapter"

In [10]:
#пропустил этот момент стал не уверен что все примеры короче 256 токенов оказалось все короче

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

dataset = load_dataset("json", data_files={"train": TRAIN_PATH, "validation": TEST_PATH})

lengths = []

for split in ["train", "validation"]:
    for inst, resp in zip(dataset[split]["instruction"], dataset[split]["response"]):
        messages = [
            {"role": "system", "content": "Ты — дружелюбный бариста-эксперт в кофейне."},
            {"role": "user", "content": inst},
            {"role": "assistant", "content": resp},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        token_ids = tokenizer(text, add_special_tokens=False)["input_ids"]
        lengths.append(len(token_ids))

lengths_sorted = sorted(lengths)
n = len(lengths_sorted)

print(f"Всего примеров: {n}")
print(f"Минимум: {lengths_sorted[0]}")
print(f"Максимум: {lengths_sorted[-1]}")
print(f"Среднее: {sum(lengths_sorted) / n:.1f}")
print(f"Медиана: {lengths_sorted[n // 2]}")
print(f"95й перцентиль: {lengths_sorted[int(n * 0.95)]}")
print(f"99й перцентиль: {lengths_sorted[int(n * 0.99)]}")

over256 = sum(1 for l in lengths if l > 256)
print(f"\nПримеров длиннее 256 токенов: {over256} ({over256 / n * 100:.1f}%)")

Всего примеров: 205
Минимум: 89
Максимум: 186
Среднее: 136.0
Медиана: 135
95й перцентиль: 164
99й перцентиль: 177

Примеров длиннее 256 токенов: 0 (0.0%)


In [2]:
bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if bf16_supported else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print("Начало загрузки токенизатора.")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Начало загрузки модели в режиме 4-битного квантования.")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

dataset = load_dataset("json", data_files={"train": TRAIN_PATH, "validation": TEST_PATH})


def format_prompts(batch):
    formatted_texts = []
    for inst, resp in zip(batch["instruction"], batch["response"]):
        messages = [
            {"role": "system", "content": "Ты — дружелюбный бариста-эксперт в кофейне."},
            {"role": "user", "content": inst},
            {"role": "assistant", "content": resp},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        formatted_texts.append(text)
    return {"text": formatted_texts}


dataset = dataset.map(
    format_prompts,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

os.environ["WANDB_PROJECT"] = "coffee-bot-finetuning"
os.environ["WANDB_LOG_MODEL"] = "false"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    eval_strategy="steps",
    eval_steps=20,
    logging_steps=5,
    learning_rate=2e-4,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=not bf16_supported,
    bf16=bf16_supported,
    save_strategy="no",
    report_to="wandb",
    run_name="qlora-qwen2.5-coffee",
    max_length=512,
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    args=training_args,
)

print("Запуск процесса дообучения модели.")
trainer.train()

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Процесс обучения завершен. Модифицированные веса сохранены в директорию {OUTPUT_DIR}")

import wandb

wandb.finish()

Начало загрузки токенизатора.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Начало загрузки модели в режиме 4-битного квантования.


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

/tmp/ipykernel_225466/1800147327.py:80: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Запуск процесса дообучения модели.


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
20,1.080230,1.124412,1.115365,0.732706,22087.000000
40,0.774192,1.037449,0.874631,0.754972,43828.000000
60,0.575018,1.038939,0.764028,0.760078,65808.000000
69,0.524220,1.043085,0.749466,0.761079,75645.000000


Процесс обучения завершен. Модифицированные веса сохранены в директорию ./coffee_qwen_adapter


eval/entropy,█▃▁▁
eval/loss,█▁▁▁
eval/mean_token_accuracy,▁▆██
eval/num_tokens,▁▄▇█
eval/runtime,█▆▁▂
eval/samples_per_second,▁▃█▇
eval/steps_per_second,▁▁▁▁
train/entropy,██▆▅▄▄▃▂▃▂▂▁▁
train/epoch,▁▂▂▃▃▃▄▄▅▅▅▆▆▇▇███
train/global_step,▁▂▂▃▃▃▄▄▅▅▅▆▆▇▇███
+5,...


 TRAIN IS OVER

TESTING


In [14]:
OUTPUT_PATH = "evaluation_results.jsonl"
MAX_NEW_TOKENS = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

test_data = []
with open(TEST_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            test_data.append(json.loads(line.strip()))

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
eos_ids = [tokenizer.eos_token_id]
if im_end_id is not None and im_end_id != tokenizer.unk_token_id and im_end_id not in eos_ids:
    eos_ids.append(im_end_id)


def generate_response(current_model, instruction):
    messages = [
        {"role": "system", "content": "Ты — дружелюбный бариста-эксперт в кофейне."},
        {"role": "user", "content": instruction},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(current_model.device)

    with torch.no_grad():
        generated_ids = current_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = [output[len(input_ids):] for input_ids, output in zip(inputs.input_ids, generated_ids)]
    return tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Загрузка базовой модели для инференса.")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

for i, item in enumerate(test_data):
    item["base_response"] = generate_response(model, item["instruction"])
    print(f"Базовая модель: {i + 1}/{len(test_data)}")



Загрузка базовой модели для инференса.


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Базовая модель: 1/21
Базовая модель: 2/21
Базовая модель: 3/21
Базовая модель: 4/21
Базовая модель: 5/21
Базовая модель: 6/21
Базовая модель: 7/21
Базовая модель: 8/21
Базовая модель: 9/21
Базовая модель: 10/21
Базовая модель: 11/21
Базовая модель: 12/21
Базовая модель: 13/21
Базовая модель: 14/21
Базовая модель: 15/21
Базовая модель: 16/21
Базовая модель: 17/21
Базовая модель: 18/21
Базовая модель: 19/21
Базовая модель: 20/21
Базовая модель: 21/21
Применение LoRA адаптера к базовой модели.


NameError: name 'ADAPTER_DIR' is not defined

In [15]:
ADAPTER_DIR = "./coffee_qwen_adapter"

In [16]:
print("Применение LoRA адаптера к базовой модели.")
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model.eval()

for i, item in enumerate(test_data):
    item["tuned_response"] = generate_response(model, item["instruction"])
    print(f"Дообученная модель: {i + 1}/{len(test_data)}")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for item in test_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Сбор ответов завершен. Файл сохранен в {OUTPUT_PATH}")

Применение LoRA адаптера к базовой модели.
Дообученная модель: 1/21
Дообученная модель: 2/21
Дообученная модель: 3/21
Дообученная модель: 4/21
Дообученная модель: 5/21
Дообученная модель: 6/21
Дообученная модель: 7/21
Дообученная модель: 8/21
Дообученная модель: 9/21
Дообученная модель: 10/21
Дообученная модель: 11/21
Дообученная модель: 12/21
Дообученная модель: 13/21
Дообученная модель: 14/21
Дообученная модель: 15/21
Дообученная модель: 16/21
Дообученная модель: 17/21
Дообученная модель: 18/21
Дообученная модель: 19/21
Дообученная модель: 20/21
Дообученная модель: 21/21
Сбор ответов завершен. Файл сохранен в evaluation_results.jsonl
